In [8]:
from dotenv import load_dotenv

import pandas as pd, os
from tqdm import tqdm
import json

In [2]:
# 환경 변수 및 설정
load_dotenv()

True

In [6]:
os.environ["LANGCHAIN_TRACING_V2"]

'true'

In [7]:
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("RFP_Development")

LangSmith 추적을 시작합니다.
[프로젝트명]
RFP_Development


DataFrame Read

In [10]:
dir_path = r"C:\kha\project\rfp\data(sample)"
f_names = [os.path.join(dir_path,f) for f in os.listdir(dir_path)]
df_list = [pd.read_csv(f, sep="\t") for f in f_names]

In [15]:
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain.agents.agent_types import AgentType
from langchain_openai import ChatOpenAI
from langchain_teddynote.messages import AgentStreamParser
import seaborn as sns

In [16]:
sns.set_style("white")

In [27]:
agent = create_pandas_dataframe_agent(
    ChatOpenAI(model="gpt-4o-mini", temperature=0),
    df_list[:4],
    verbose=False,
    agent_type=AgentType.OPENAI_FUNCTIONS,
    allow_dangerous_code=True,
)

In [28]:
stream_parser = AgentStreamParser()

In [29]:
def ask(query):
    # 질의에 대한 답변을 출력합니다.
    response = agent.stream({"input": query})

    for step in response:
        stream_parser.process_agent_steps(step)

In [30]:
ask("몇 개의 행이 있어?")

[도구 호출]
Tool: python_repl_ast
query: len(df1), len(df2), len(df3), len(df4)
Log: 
Invoking: `python_repl_ast` with `{'query': 'len(df1), len(df2), len(df3), len(df4)'}`



[관찰 내용]
Observation: (1188, 1085, 1085, 44528)
[최종 답변]
각 데이터프레임의 행 수는 다음과 같습니다:

- df1: 1188행
- df2: 1085행
- df3: 1085행
- df4: 44528행


In [31]:
ask("한 개의 bidding에 대해 몇개의 요구사항이 있어?")

[최종 답변]
각 bidding에 대한 요구사항의 수를 확인하기 위해서는 `df4` 데이터프레임을 사용해야 합니다. 이 데이터프레임은 요구사항 정보를 포함하고 있습니다. 각 bidding에 대한 요구사항의 수를 세기 위해 `bidding_info_id`를 기준으로 그룹화하고, 각 그룹의 크기를 계산하면 됩니다.

다음 코드를 사용하여 각 bidding에 대한 요구사항의 수를 확인할 수 있습니다:

```python
requirement_counts = df4.groupby('bidding_info_id').size()
requirement_counts
```

이 코드를 실행하면 각 bidding에 대해 몇 개의 요구사항이 있는지 확인할 수 있습니다.


In [32]:
ask("기초학력 진단과 관련된 사업을 관련된 순으로 10개만 알려줘")

[도구 호출]
Tool: python_repl_ast
query: import pandas as pd

# Assuming df1, df2, df3, df4 are already defined as per the provided data

# Searching for '기초학력' in df1, df2, df3, and df4
related_projects = []

# Check df1 for related projects
for index, row in df1.iterrows():
    if '기초학력' in row['name'] or '기초학력' in row['summary']:
        related_projects.append(row)

# Check df2 for related projects
for index, row in df2.iterrows():
    if '기초학력' in row['summary']:
        related_projects.append(row)

# Check df3 for related projects
for index, row in df3.iterrows():
    if '기초학력' in row['summary']:
        related_projects.append(row)

# Check df4 for related projects
for index, row in df4.iterrows():
    if '기초학력' in row['summary']:
        related_projects.append(row)

# Limit to 10 related projects
related_projects = related_projects[:10]
related_projects
Log: 
Invoking: `python_repl_ast` with `{'query': "import pandas as pd\n\n# Assuming df1, df2, df3, df4 are already defined as p